# 09 — Avatar export for a few TEST sentences

**Purpose.** After the single test evaluation: motion JSON (for the browser VRM viewer) and side-by-side skeleton videos (Model B vs B1 vs the real recording) for 5 test sentences. Nothing is re-scored.

**Kaggle settings.** Accelerator **None** · Internet **On** (pip install OpenCV if missing).

**Inputs.** Dataset `bdsl-k400` + **Add Input → Output of notebook 08**.

**Outputs** (under `/kaggle/working/k400/`, saved by *Save Version → Save & Run All*). `outputs/avatar/<Index>/` (motion_modelB.json, motion_B1.json, motion_real.json, demo_log.json, skeleton_modelB_B1_real.mp4, info.json, LIMITATIONS.md).

**Expected run time** [ESTIMATE, not measured yet]. under 5 minutes.

**What to check.** Refuses to run without `FINAL_TEST_LOCK.json` (correct). Five folders written; the video plays below.

> Honesty rules: label every number measured or estimated; results are small-scale and signer-dependent;
> the test split is used once (notebook 07); low coordinate error is not proof of correct sign language;
> G0 glosses are pseudo-glosses, not BdSL annotation; no human or expert validation exists.

In [ ]:
# Cell 1 - Setup (the same in every notebook). Finds the data and the code, sets the paths, and copies the
# outputs of earlier notebooks (added with "Add Input") into /kaggle/working/k400 so results accumulate.
import os, sys, zipfile, pathlib, json
NOTEBOOK = "09_avatar_export"

def _find_root():
    base = pathlib.Path("/kaggle/input")
    if base.is_dir():
        for dirpath, dirnames, filenames in os.walk(base):
            if "UPLOAD_MANIFEST.csv" in filenames and "PACKAGE_INFO.json" in filenames:
                return pathlib.Path(dirpath)
            depth = len(pathlib.Path(dirpath).relative_to(base).parts)
            dirnames[:] = [] if depth >= 4 else [d for d in dirnames if d not in ("videos", "k400")]
        zips = sorted(base.rglob("bdsl-k400-*.zip"))
        if zips:  # Kaggle kept the upload as one zip file: unpack it once (a few minutes)
            dest = pathlib.Path("/tmp/k400_dataset")   # /tmp is not saved as Output
            if not (dest / "UPLOAD_MANIFEST.csv").is_file():
                print("Unpacking", zips[0], "->", dest)
                zipfile.ZipFile(zips[0]).extractall(dest)
            return dest
        raise FileNotFoundError("The private dataset is not attached. Right panel: Add Input -> Your Datasets -> bdsl-k400.")
    here = pathlib.Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "pyproject.toml").is_file() and (c / "kaggle" / "src" / "k400").is_dir():
            return c
    raise FileNotFoundError("Open this notebook inside the project folder, or on Kaggle with the dataset attached.")

DATASET_ROOT = _find_root()
CODE_ROOT = DATASET_ROOT / "code" if (DATASET_ROOT / "code" / "pyproject.toml").is_file() else DATASET_ROOT
for _p in (CODE_ROOT / "kaggle" / "src", CODE_ROOT / "src"):
    sys.path.insert(0, str(_p))
from k400.setup import setup
S = setup(DATASET_ROOT, CODE_ROOT, NOTEBOOK)

In [ ]:
try:
    import cv2  # noqa: F401
except ImportError:
    S.pip_install("opencv-contrib-python==5.0.0.93")
assert (S.state_root / "outputs/FINAL_TEST_LOCK.json").is_file(), "Run notebook 07 first (and add its output as Input)"
rc_avatar = S.k400("avatar")

In [ ]:
from IPython.display import Video, display
import glob
vids = sorted(glob.glob(str(S.state_root / "outputs/avatar/*/skeleton_modelB_B1_real.mp4")))
print(len(vids), "videos")
if vids:
    display(Video(vids[0], embed=True, width=720))

In [ ]:
S.finish(checks=[
    ("avatar export done", rc_avatar == 0),
    ("5 skeleton videos", len(vids) == 5),
])